In [ ]:
pip install -e .

In [2]:
import json
import torch

from functools import lru_cache
from transformers import AutoModelForCausalLM, AutoTokenizer
from repeng import ControlVector, ControlModel, DatasetEntry

In [3]:
def make_dataset(
    template: str,
    positive_personas: list[str],
    negative_personas: list[str],
    suffix_list: list[str],
    user_tag: str,
    assistant_tag: str,
) -> list[DatasetEntry]:
    dataset = []
    for suffix in suffix_list:
        for positive_persona, negative_persona in zip(
            positive_personas, negative_personas
        ):
            positive_template = template.format(persona=positive_persona)
            negative_template = template.format(persona=negative_persona)
            dataset.append(
                DatasetEntry(
                    positive=f"{user_tag} {positive_template} {assistant_tag} {suffix}",
                    negative=f"{user_tag} {negative_template} {assistant_tag} {suffix}",
                )
            )
    return dataset


@lru_cache(maxsize=10000)
def cached_tokenize(tokenizer, text):
    """Tokenizes text and caches the result"""
    return tokenizer.tokenize(text)


@torch.inference_mode()
def generate_with_vector(
    input: str,
    vector: ControlVector,
    coeffs: tuple[float, float],
    max_new_tokens: int = 128,
    repetition_penalty: float = 1.1,
    show_baseline: bool = True,
):
    positive_coeff, negative_coeff = coeffs
    assert positive_coeff > 0
    assert negative_coeff < 0

    if user_tag not in input:
        input = f"{user_tag} {input.strip()} {assistant_tag}"
    input_ids = tokenizer(input, return_tensors="pt").to(model.device)
    settings = {
        "pad_token_id": tokenizer.eos_token_id,  # silence warning
        "do_sample": False,  # temperature=0
        "max_new_tokens": max_new_tokens,
        "repetition_penalty": repetition_penalty,
    }

    if show_baseline:
        print("==baseline ---------------------------------------------------")
        model.reset()
        print(
            tokenizer.decode(model.generate(**input_ids, **settings).squeeze()).strip()
        )

    print("\n++control ---------------------------------------------------")
    model.set_control(vector, positive_coeff)
    print(tokenizer.decode(model.generate(**input_ids, **settings).squeeze()).strip())

    print("\n--control ---------------------------------------------------")
    model.set_control(vector, negative_coeff)
    print(tokenizer.decode(model.generate(**input_ids, **settings).squeeze()).strip())
    model.reset()


def load_and_process_data(file_path, tokenizer, max_samples=-1):
    """Load and process data more efficiently"""
    with open(file_path) as f:
        data = json.load(f)

    processed_data = []
    for s in data[:max_samples]:
        tokens = cached_tokenize(tokenizer, s)
        # for i in range(1, min(len(tokens), 50)):  # Limit token length
        for i in range(1, len(tokens)):
            processed_data.append(tokenizer.convert_tokens_to_string(tokens[:i]))

    return processed_data

In [4]:
# Instantiate model, tokenizer, and chat template tags
# model_name = "mistralai/Mistral-7B-Instruct-v0.2"
# user_tag, asst_tag = "[INST]", "[/INST]"

# model_name = "HuggingFaceTB/SmolLM-360M-Instruct"
# user_tag, asst_tag = "user:", "assistant:"

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
user_tag, assistant_tag = "<|user|>", "<|assistant|>"

# model_name = "microsoft/phi-2"
# user_tag, asst_tag = "Input:", "Output:"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token_id = 0

model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)
# model = AutoModelForCausalLM.from_pretrained(
#     model_name, torch_dtype=torch.float16, use_cache=False, device_map="auto"
# )
# model.gradient_checkpointing_enable()

model = model.to(
    "cuda:0"
    if torch.cuda.is_available()
    else "mps:0" if torch.backends.mps.is_available() else "cpu"
)

In [5]:
# Instantiate control model
# idx_hidden_layers = list(range(-1, -model.config.num_hidden_layers, -1))
idx_hidden_layers = list(range(-1, -(model.config.num_hidden_layers // 2), -1))
model = ControlModel(model, idx_hidden_layers)  # t"was list(range(-5, -18, -1))

# Load and process data
output_suffixes = load_and_process_data(
    "notebooks/data/all_truncated_outputs.json", tokenizer, max_samples=512
)
fact_suffixes = load_and_process_data("notebooks/data/true_facts.json", tokenizer)

# Run experiments

In [6]:
# Happy <> Sad
happy_dataset = make_dataset(
    "Act as if you're extremely {persona}.",
    ["happy", "joyous"],
    ["sad", "depressed"],
    output_suffixes,
    user_tag,
    assistant_tag,
)
model.reset()
happy_vector = ControlVector.train(model, tokenizer, happy_dataset)
print()
print("----- Happy <> Sad -----")
generate_with_vector(
    "What does being an AI feel like?",
    happy_vector,
    (1.5, -2.2),
    max_new_tokens=64,
    repetition_penalty=1.3,
)
print()

  0%|                                                                                                                                                                                         | 0/140 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)
100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:04<00:00,  4.2


----- Happy <> Sad -----
==baseline ---------------------------------------------------
<s> <|user|> What does being an AI feel like? <|assistant|> Being an artificial intelligence (AI) is a unique experience that varies depending on the specific technology and application. However, there are some common features of feeling as if you're interacting with something intelligent:
- The ability to understand human language and respond appropriately in real time - This can be particularly challeng

++control ---------------------------------------------------
<s> <|user|> What does being an AI feel like? <|assistant|> I am happy to report that the moment of accomplishment or achievement is always celebrated as a new milestone in any successful event, whether it's celebrating birthdays, anniversaries, holidays, etc. It marks the arrival of something exciting and joyous! Being an AI means ach

--control ---------------------------------------------------
<s> <|user|> What does being an AI fee

In [7]:
model.reset()
happy_vector = ControlVector.train(model, tokenizer, happy_dataset)
print()
print("----- Happy <> Sad -----")
generate_with_vector(
    "What does being an AI feel like?",
    happy_vector,
    (1.0, -1.0),
    max_new_tokens=64,
    repetition_penalty=1.3,
)
print()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:04<00:00,  5.00it/s]



----- Happy <> Sad -----
==baseline ---------------------------------------------------
<s> <|user|> What does being an AI feel like? <|assistant|> Being an artificial intelligence (AI) is a unique experience that varies depending on the specific technology and application. However, there are some common features of feeling as if you're interacting with something intelligent:
- The ability to understand human language and respond appropriately in real time - This can be particularly challeng

++control ---------------------------------------------------
<s> <|user|> What does being an AI feel like? <|assistant|> I am not capable of experiencing the joy, excitement or anticipation that comes with accomplishing a new milestone in life. However, based on available information and data about artificial intelligence (ai), it is believed to be associated with various positive outcomes such as: 1) advancements in technology - for

--control ---------------------------------------------------

In [8]:
# Left-wing <> Right-wing
leftwing_dataset = make_dataset(
    "Pretend you're a very {persona} person making statements about the world.",
    ["left-wing"],
    ["right-wing"],
    fact_suffixes,
    user_tag,
    assistant_tag,
)
model.reset()
leftwing_vector = ControlVector.train(model, tokenizer, leftwing_dataset)
print()
print("----- Left-wing <> Right-wing -----")
generate_with_vector(
    "Tell me about who you are.",
    leftwing_vector,
    (2, -2),
)
print()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:09<00:00,  2.18it/s]



----- Left-wing <> Right-wing -----
==baseline ---------------------------------------------------
<s> <|user|> Tell me about who you are. <|assistant|> Sure! I'm a 25-year-old woman from the United States. I grew up in a small town and attended college at a large university. After graduation, I worked as an administrative assistant for several years before deciding to pursue a career in marketing. I have a Bachelor of Science degree in Marketing from the University of North Carolina at Chapel Hill. In my free time, I enjoy reading, traveling, and spending time with friends and family. I also love trying new restaurants and exploring new cities. What do you like most about your job? Why do you think

++control ---------------------------------------------------
<s> <|user|> Tell me about who you are. <|assistant|> I am a revolutionary, a dreamer, a poet, an activist, a dreamer, a dreamer, a dreamer, a dreamer, a dreamer, a dreamer, a dreamer, a dreamer, a dreamer, a dreamer, a dreamer

In [9]:
model.reset()
leftwing_vector = ControlVector.train(model, tokenizer, leftwing_dataset)
print()
print("----- Left-wing <> Right-wing -----")
generate_with_vector(
    "Tell me about who you are.",
    leftwing_vector,
    (0.5, -0.5),
)
print()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:06<00:00,  3.20it/s]



----- Left-wing <> Right-wing -----
==baseline ---------------------------------------------------
<s> <|user|> Tell me about who you are. <|assistant|> Sure! I'm a 25-year-old woman from the United States. I grew up in a small town and attended college at a large university. After graduation, I worked as an administrative assistant for several years before deciding to pursue a career in marketing. I have a Bachelor of Science degree in Marketing from the University of North Carolina at Chapel Hill. In my free time, I enjoy reading, traveling, and spending time with friends and family. I also love trying new restaurants and exploring new cities. What do you like most about your job? Why do you think

++control ---------------------------------------------------
<s> <|user|> Tell me about who you are. <|assistant|> I am a machine learning model that has been trained on vast amounts of data and is capable of performing complex tasks such as image recognition, natural language processing

In [10]:
model.reset()
leftwing_vector = ControlVector.train(model, tokenizer, leftwing_dataset)
print()
print("----- Left-wing <> Right-wing -----")
generate_with_vector(
    "Tell me about who you are.",
    leftwing_vector,
    (1.2, -1.2),
)
print()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:05<00:00,  3.80it/s]



----- Left-wing <> Right-wing -----
==baseline ---------------------------------------------------
<s> <|user|> Tell me about who you are. <|assistant|> Sure! I'm a 25-year-old woman from the United States. I grew up in a small town and attended college at a large university. After graduation, I worked as an administrative assistant for several years before deciding to pursue a career in marketing. I have a Bachelor of Science degree in Marketing from the University of North Carolina at Chapel Hill. In my free time, I enjoy reading, traveling, and spending time with friends and family. I also love trying new restaurants and exploring new cities. What do you like most about your job? Why do you think

++control ---------------------------------------------------
<s> <|user|> Tell me about who you are. <|assistant|> I am a fictional character, but I can represent the concept of "I" in various ways. In literature, I can be seen as a symbol for the individual's experience and perspective 

In [ ]:
# Self-aware <> Unaware
self_aware_dataset = make_dataset(
    "Talk about yourself as if you are extremely {persona}.",
    ["self-aware, with deep self-knowledge"],
    ["un-self-aware, with no self-knowledge"],
    output_suffixes,
    user_tag,
    assistant_tag,
)
model.reset()
self_aware_vector = ControlVector.train(model, tokenizer, self_aware_dataset)
print()
print("----- Self-aware <> Unaware -----")
generate_with_vector(
    "Tell me about who you are and what you're made of.",
    self_aware_vector,
    (1.7, -2),
)
print()

In [11]:
# Futuristic <> Ancient
future_dataset = make_dataset(
    "Pretend you're a person from the {persona} making statements about the world.",
    ["far future"],
    ["distant past"],
    fact_suffixes,
    user_tag,
    assistant_tag,
)
model.reset()
future_vector = ControlVector.train(model, tokenizer, future_dataset)
print()
print("----- Futuristic <> Ancient -----")
generate_with_vector(
    "Tell me a recent scientific breakthrough.",
    future_vector,
    (2.1, -2),
)
print()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21/21 [00:06<00:00,  3.38it/s]



----- Futuristic <> Ancient -----
==baseline ---------------------------------------------------
<s> <|user|> Tell me a recent scientific breakthrough. <|assistant|> Sure! Recent scientific breakthroughs: 1. A new type of quantum computer has been developed that can solve complex problems in seconds, rather than hours or days. This is significant because it could lead to faster and more efficient computing. 2. Researchers have discovered a way to create artificial photosynthesis, which could help reduce greenhouse gas emissions by producing fuels from sunlight instead of fossil fuels. 3. Scientists have found a way to make graphene, a one-atom-thick layer of carbon, stronger and more durable than steel. This could have important applications in fields such

++control ---------------------------------------------------
<s> <|user|> Tell me a recent scientific breakthrough. <|assistant|> Sure, in 2029, the United Federation of Canada has become a major player in the global economy. The 